# Capstone Project: Customer Churn Prediction & Retention Strategy
### Internship Track: Week 4 – Task 1
**Author:** Sayantani Adhikary  
**Objective:** Solve a real-world subscription churn problem end-to-end using EDA, feature engineering, model comparison (Decision Tree, Logistic Regression, Random Forest), and actionable business recommendations.


## 1. Import Libraries & Load Dataset


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

sns.set_theme(style="whitegrid")
df = pd.read_csv('../data/customer_churn_capstone.csv')
print(f"Dataset Shape: {df.shape}")
df.head()


## 2. Exploratory Data Analysis (EDA)
Visualize churn rate by `ContractType` and inspect the relationship between `TenureMonths`, `MonthlyCharges`, and `Churn`.


In [1]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

churn_by_c = df.groupby('ContractType')['Churn'].mean().reset_index()
churn_by_c['ChurnPct'] = churn_by_c['Churn'] * 100
sns.barplot(x='ContractType', y='ChurnPct', data=churn_by_c,
            order=['Month-to-month', 'One year', 'Two year'], palette='Set2', ax=axes[0])
axes[0].set_title('Churn Rate by Contract Type (%)', fontweight='bold')
axes[0].set_ylabel('Churn Rate (%)')

sns.scatterplot(x='TenureMonths', y='MonthlyCharges', hue='Churn', data=df,
                palette={0: '#2b6cb0', 1: '#e53e3e'}, s=80, ax=axes[1])
axes[1].set_title('Tenure vs. Monthly Charges by Churn', fontweight='bold')
plt.tight_layout()
plt.show()


## 3. Feature Engineering & Preprocessing
1. Engineer `AvgMonthlySpend` and `HighTicketRiskFlag` (`SupportTickets >= 3`).
2. One-Hot Encode `ContractType`.
3. Split 80/20 and standardize numerical features with `StandardScaler`.


In [1]:
df['AvgMonthlySpend'] = (df['TotalCharges'] / df['TenureMonths']).round(2)
df['HighTicketRiskFlag'] = (df['SupportTickets'] >= 3).astype(int)

X = df.drop(columns=['CustomerID', 'Churn'])
y = df['Churn']

X_encoded = pd.get_dummies(X, columns=['ContractType'], drop_first=True, dtype=int)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Preprocessing and 80/20 split completed!")


## 4. Model Comparison & Performance Summary
Train and evaluate **Decision Tree**, **Logistic Regression**, and **Random Forest**.


In [1]:
models = {
    'Decision Tree': DecisionTreeClassifier(max_depth=4, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
}

summary = []
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    summary.append({
        'Model': name,
        'Accuracy (%)': round(accuracy_score(y_test, preds) * 100, 2),
        'Precision (%)': round(precision_score(y_test, preds, zero_division=0) * 100, 2),
        'Recall (%)': round(recall_score(y_test, preds, zero_division=0) * 100, 2),
        'F1-Score (%)': round(f1_score(y_test, preds, zero_division=0) * 100, 2)
    })

summary_df = pd.DataFrame(summary)
print("=== Model Performance Summary ===")
print(summary_df.to_string(index=False))


## 5. Business Recommendations
1. **Annual Contract Incentives**: Offer 15% upgrade discounts to Month-to-month subscribers before month 6.
2. **Proactive Support Escalation**: Flag accounts with 3+ support tickets for immediate outreach.
3. **First-Year Onboarding**: Focus retention check-ins on customers in their first 12 months of tenure.
